In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("SahlPay-Partie3-DataFrames")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.executor.memory", "1g")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.jars.ivy", "/tmp/.ivy2")
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.3.4,"
            "com.amazonaws:aws-java-sdk-bundle:1.12.262")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "sahlpay")
    .config("spark.hadoop.fs.s3a.secret.key", "sahlpay123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)
sc = spark.sparkContext
print(spark.version, sc.master, sc.defaultParallelism)

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /tmp/.ivy2/cache
The jars for the packages stored in: /tmp/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-aada3ea5-5156-4486-8b94-fc92bf78d4f3;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 282ms :: artifacts dl 5ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwn

3.5.5 spark://spark-master:7077 2


In [2]:
%%writefile sahlpay_schemas.py
from pyspark.sql.types import (StructType, StructField, StringType,
                               DoubleType, TimestampType)

tx_schema = StructType([
    StructField("tx_id",        StringType(),    True),
    StructField("type",         StringType(),    True),
    StructField("payer_wallet", StringType(),    True),
    StructField("payee_id",     StringType(),    True),
    StructField("device_id",    StringType(),    True),
    StructField("city",         StringType(),    True),
    StructField("channel",      StringType(),    True),
    StructField("status",       StringType(),    True),
    StructField("amount_mad",   DoubleType(),    True),
    StructField("event_time",   TimestampType(), True),
])

merchants_schema = StructType([
    StructField("merchant_id", StringType(), True),
    StructField("name",        StringType(), True),
    StructField("city",        StringType(), True),
    StructField("category",    StringType(), True),
    StructField("latitude",    DoubleType(), True),
    StructField("longitude",   DoubleType(), True),
])

Writing sahlpay_schemas.py


In [3]:
from sahlpay_schemas import tx_schema, merchants_schema
from pyspark.sql import functions as F
import time, json

PATH_TX = "s3a://lake/bronze/history/transactions_history.jsonl"
PATH_M  = "s3a://lake/bronze/reference/merchants.json"

tx = spark.read.schema(tx_schema).json(PATH_TX)
merchants = spark.read.schema(merchants_schema).json(PATH_M)

tx.printSchema()
print("transactions :", tx.count(), "| merchants :", merchants.count())
tx.show(3, truncate=False)

26/11/20 14:51:17 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


root
 |-- tx_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- payer_wallet: string (nullable = true)
 |-- payee_id: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- status: string (nullable = true)
 |-- amount_mad: double (nullable = true)
 |-- event_time: timestamp (nullable = true)



transactions : 50000 | merchants : 300
+-----------+----+------------+--------+---------+----+-------+----------+----------+-------------------+
|tx_id      |type|payer_wallet|payee_id|device_id|city|channel|status    |amount_mad|event_time         |
+-----------+----+------------+--------+---------+----+-------+----------+----------+-------------------+
|TX-00016455|P2M |W-10158     |M-046   |D-370bc0 |Fes |QR     |INITIATED |104.08    |2026-11-15 23:02:16|
|TX-00016455|P2M |W-10158     |M-046   |D-370bc0 |Fes |QR     |AUTHORIZED|104.08    |2026-11-15 23:02:18|
|TX-00016455|P2M |W-10158     |M-046   |D-370bc0 |Fes |QR     |COMPLETED |104.08    |2026-11-15 23:02:22|
+-----------+----+------------+--------+---------+----+-------+----------+----------+-------------------+
only showing top 3 rows



In [4]:
completed_df = tx.filter(F.col("status") == "COMPLETED").cache()
print("COMPLETED :", completed_df.count())     # 15525 attendu

def city_df_job():
    return (completed_df.groupBy("city")
            .agg(F.sum("amount_mad").alias("total_mad"),
                 F.count("*").alias("nb"),
                 F.avg("amount_mad").alias("avg_mad"))
            .orderBy(F.desc("total_mad")))

city_df = city_df_job()
city_df.show(truncate=False)

COMPLETED : 15525


[Stage 11:>                                                         (0 + 3) / 3]

+----------+------------------+----+------------------+
|city      |total_mad         |nb  |avg_mad           |
+----------+------------------+----+------------------+
|Casablanca|899587.45         |4367|205.99666819326768|
|Rabat     |582048.6299999995 |2893|201.19206014517786|
|Marrakech |454522.3900000001 |2041|222.69592846643806|
|Fes       |242935.21999999997|1336|181.83773952095805|
|Tanger    |232327.27000000002|1141|203.6172392638037 |
|Agadir    |180523.61000000002|1101|163.96331516802908|
|Kenitra   |133977.89         |709 |188.96740479548663|
|Oujda     |128017.23999999996|691 |185.2637337192474 |
|Meknes    |117299.19         |676 |173.51951183431953|
|Tetouan   |104792.05         |570 |183.84570175438597|
+----------+------------------+----+------------------+



In [5]:
ref = {  # valeurs obtenues avec les RDD en Partie 2
    "Casablanca": (899587.45, 4367), "Rabat": (582048.63, 2893),
    "Marrakech": (454522.39, 2041), "Fes": (242935.22, 1336),
    "Tanger": (232327.27, 1141), "Agadir": (180523.61, 1101),
    "Kenitra": (133977.89, 709), "Oujda": (128017.24, 691),
    "Meknes": (117299.19, 676), "Tetouan": (104792.05, 570),
}
for r in city_df.collect():
    tot, nb = ref[r["city"]]
    assert abs(r["total_mad"] - tot) < 0.01 and r["nb"] == nb, r
print("OK : les résultats DataFrame = résultats RDD de la Partie 2")

OK : les résultats DataFrame = résultats RDD de la Partie 2


In [6]:
completed_df.createOrReplaceTempView("tx_completed")
tx.createOrReplaceTempView("transactions")

sql_city = spark.sql("""
    SELECT city,
           ROUND(SUM(amount_mad), 2) AS total_mad,
           COUNT(*)                  AS nb,
           ROUND(AVG(amount_mad), 2) AS avg_mad
    FROM tx_completed
    GROUP BY city
    ORDER BY total_mad DESC
""")
sql_city.show(truncate=False)

+----------+---------+----+-------+
|city      |total_mad|nb  |avg_mad|
+----------+---------+----+-------+
|Casablanca|899587.45|4367|206.0  |
|Rabat     |582048.63|2893|201.19 |
|Marrakech |454522.39|2041|222.7  |
|Fes       |242935.22|1336|181.84 |
|Tanger    |232327.27|1141|203.62 |
|Agadir    |180523.61|1101|163.96 |
|Kenitra   |133977.89|709 |188.97 |
|Oujda     |128017.24|691 |185.26 |
|Meknes    |117299.19|676 |173.52 |
|Tetouan   |104792.05|570 |183.85 |
+----------+---------+----+-------+



In [7]:
m = merchants.select(
    F.col("merchant_id"),
    F.col("name").alias("merchant_name"),
    F.col("city").alias("merchant_city"),
    F.col("category"),
)

p2m = completed_df.filter(F.col("type") == "P2M")

# contrôle : aucun paiement P2M sans commerçant correspondant
print("P2M sans commerçant :",
      p2m.join(m, p2m.payee_id == m.merchant_id, "left_anti").count())   # 0 attendu

joined = p2m.join(m, p2m.payee_id == m.merchant_id, "inner")

by_cat_city = (joined.groupBy("category", "merchant_city")
               .agg(F.round(F.sum("amount_mad"), 2).alias("total_mad"),
                    F.count("*").alias("nb"))
               .orderBy(F.desc("total_mad")))
by_cat_city.show(20, truncate=False)

# vue d'ensemble par catégorie
joined.groupBy("category").agg(F.round(F.sum("amount_mad"), 2).alias("total_mad"),
                               F.count("*").alias("nb")).orderBy(F.desc("total_mad")).show()

P2M sans commerçant : 0
+------------+-------------+---------+---+
|category    |merchant_city|total_mad|nb |
+------------+-------------+---------+---+
|carburant   |Casablanca   |137628.49|717|
|telecom     |Casablanca   |136098.58|698|
|epicerie    |Casablanca   |119308.42|617|
|restauration|Casablanca   |113434.96|533|
|pharmacie   |Casablanca   |100891.22|539|
|ecommerce   |Casablanca   |80228.28 |445|
|restauration|Rabat        |79320.52 |453|
|telecom     |Rabat        |78562.24 |447|
|epicerie    |Rabat        |77811.46 |433|
|ecommerce   |Marrakech    |77270.61 |421|
|pharmacie   |Marrakech    |77035.63 |379|
|ecommerce   |Rabat        |68411.84 |373|
|pharmacie   |Rabat        |68360.49 |363|
|epicerie    |Fes          |66013.87 |406|
|ecommerce   |Fes          |64335.38 |329|
|telecom     |Marrakech    |63727.34 |313|
|restauration|Tanger       |48889.41 |209|
|epicerie    |Oujda        |44877.0  |252|
|carburant   |Rabat        |39054.09 |237|
|restauration|Marrakech    |37

In [8]:
from pyspark.sql.types import StringType

def amount_band(a):
    if a is None:
        return None
    if a < 50:
        return "micro"
    if a <= 2000:
        return "standard"
    return "eleve"

amount_band_udf = F.udf(amount_band, StringType())
spark.udf.register("amount_band", amount_band, StringType())   # utilisable aussi en SQL

band_native = (F.when(F.col("amount_mad").isNull(), F.lit(None).cast("string"))
                .when(F.col("amount_mad") < 50, "micro")
                .when(F.col("amount_mad") <= 2000, "standard")
                .otherwise("eleve"))

df_udf    = tx.withColumn("amount_band", amount_band_udf("amount_mad"))
df_native = tx.withColumn("amount_band", band_native)

# les deux versions doivent donner exactement les mêmes résultats
diff = (df_udf.select("tx_id", "status", F.col("amount_band").alias("a"))
        .join(df_native.select("tx_id", "status", F.col("amount_band").alias("b")),
              ["tx_id", "status"])
        .filter(F.col("a") != F.col("b")).count())
print("Différences UDF / natif :", diff)       # 0 attendu
df_native.groupBy("amount_band").count().show()

Différences UDF / natif : 0
+-----------+-----+
|amount_band|count|
+-----------+-----+
|      micro|12977|
|   standard|36854|
|      eleve|  169|
+-----------+-----+



In [9]:
import statistics

def bench(fn, runs=5):
    fn()                                  # exécution de chauffe
    ts = []
    for _ in range(runs):
        t0 = time.time(); fn(); ts.append(time.time() - t0)
    return statistics.median(ts), ts

run_udf    = lambda: df_udf.write.format("noop").mode("overwrite").save()
run_native = lambda: df_native.write.format("noop").mode("overwrite").save()

med_udf, _    = bench(run_udf)
med_native, _ = bench(run_native)
print(f"UDF Python        : {med_udf:.2f} s (médiane de 5)")
print(f"when/otherwise    : {med_native:.2f} s (médiane de 5)")
print(f"Rapport           : x{med_udf / med_native:.1f}")

df_udf.explain()
df_native.explain()

UDF Python        : 0.49 s (médiane de 5)
when/otherwise    : 0.38 s (médiane de 5)
Rapport           : x1.3
== Physical Plan ==
*(1) Project [tx_id#0, type#1, payer_wallet#2, payee_id#3, device_id#4, city#5, channel#6, status#7, amount_mad#8, event_time#9, pythonUDF0#2561 AS amount_band#2212]
+- BatchEvalPython [amount_band(amount_mad#8)#2211], [pythonUDF0#2561]
   +- FileScan json [tx_id#0,type#1,payer_wallet#2,payee_id#3,device_id#4,city#5,channel#6,status#7,amount_mad#8,event_time#9] Batched: false, DataFilters: [], Format: JSON, Location: InMemoryFileIndex(1 paths)[s3a://lake/bronze/history/transactions_history.jsonl], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<tx_id:string,type:string,payer_wallet:string,payee_id:string,device_id:string,city:string,...


== Physical Plan ==
*(1) Project [tx_id#0, type#1, payer_wallet#2, payee_id#3, device_id#4, city#5, channel#6, status#7, amount_mad#8, event_time#9, CASE WHEN isnull(amount_mad#8) THEN null WHEN (amount_mad#8 < 5

In [10]:
clean = (tx.dropDuplicates(["tx_id", "status"])
         .withColumn("amount_band", band_native))

silver = (clean
          .join(F.broadcast(m.select("merchant_id", F.col("category").alias("merchant_category"))),
                clean.payee_id == F.col("merchant_id"), "left")
          .drop("merchant_id"))

print("Avant :", tx.count(), "| après dédoublonnage :", clean.count())

OUT = "s3a://lake/silver/transactions_history/"
(silver.write.mode("overwrite")
       .partitionBy("city")
       .parquet(OUT))

back = spark.read.parquet(OUT)
print("Relecture Parquet :", back.count())
back.printSchema()

Avant : 50000 | après dédoublonnage : 49003


Relecture Parquet : 49003
root
 |-- tx_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- payer_wallet: string (nullable = true)
 |-- payee_id: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- status: string (nullable = true)
 |-- amount_mad: double (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- amount_band: string (nullable = true)
 |-- merchant_category: string (nullable = true)
 |-- city: string (nullable = true)



In [11]:
spark.read.parquet(OUT).filter(F.col("city") == "Rabat").explain()

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [tx_id#2794,type#2795,payer_wallet#2796,payee_id#2797,device_id#2798,channel#2799,status#2800,amount_mad#2801,event_time#2802,amount_band#2803,merchant_category#2804,city#2805] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[s3a://lake/silver/transactions_history], PartitionFilters: [isnotnull(city#2805), (city#2805 = Rabat)], PushedFilters: [], ReadSchema: struct<tx_id:string,type:string,payer_wallet:string,payee_id:string,device_id:string,channel:stri...




In [12]:
q = (p2m.join(m, p2m.payee_id == m.merchant_id, "inner")
        .groupBy("category", "merchant_city")
        .agg(F.sum("amount_mad").alias("total_mad")))
q.explain(True)

== Parsed Logical Plan ==
'Aggregate ['category, 'merchant_city], ['category, 'merchant_city, sum('amount_mad) AS total_mad#2862]
+- Join Inner, (payee_id#3 = merchant_id#20)
   :- Filter (type#1 = P2M)
   :  +- Filter (status#7 = COMPLETED)
   :     +- Relation [tx_id#0,type#1,payer_wallet#2,payee_id#3,device_id#4,city#5,channel#6,status#7,amount_mad#8,event_time#9] json
   +- Project [merchant_id#20, name#21 AS merchant_name#1298, city#22 AS merchant_city#1299, category#23]
      +- Relation [merchant_id#20,name#21,city#22,category#23,latitude#24,longitude#25] json

== Analyzed Logical Plan ==
category: string, merchant_city: string, total_mad: double
Aggregate [category#23, merchant_city#1299], [category#23, merchant_city#1299, sum(amount_mad#8) AS total_mad#2862]
+- Join Inner, (payee_id#3 = merchant_id#20)
   :- Filter (type#1 = P2M)
   :  +- Filter (status#7 = COMPLETED)
   :     +- Relation [tx_id#0,type#1,payer_wallet#2,payee_id#3,device_id#4,city#5,channel#6,status#7,amount_ma

In [13]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
spark.conf.set("spark.sql.adaptive.enabled", "false")
q.explain()                      # doit montrer SortMergeJoin + 2 Exchange + 2 Sort
q.collect()

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10485760")
spark.conf.set("spark.sql.adaptive.enabled", "true")

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[category#23, merchant_city#1299], functions=[sum(amount_mad#8)])
   +- Exchange hashpartitioning(category#23, merchant_city#1299, 200), ENSURE_REQUIREMENTS, [plan_id=1672]
      +- HashAggregate(keys=[category#23, merchant_city#1299], functions=[partial_sum(amount_mad#8)])
         +- Project [amount_mad#8, merchant_city#1299, category#23]
            +- BroadcastHashJoin [payee_id#3], [merchant_id#20], Inner, BuildRight, false
               :- Project [payee_id#3, amount_mad#8]
               :  +- Filter ((isnotnull(type#1) AND (type#1 = P2M)) AND isnotnull(payee_id#3))
               :     +- InMemoryTableScan [amount_mad#8, payee_id#3, type#1], [isnotnull(type#1), (type#1 = P2M), isnotnull(payee_id#3)]
               :           +- InMemoryRelation [tx_id#0, type#1, payer_wallet#2, payee_id#3, device_id#4, city#5, channel#6, status#7, amount_mad#8, event_time#9], StorageLevel(disk, memory, deserialized,

In [14]:
rdd_completed = (sc.textFile(PATH_TX)
                 .map(json.loads)
                 .filter(lambda t: t["status"] == "COMPLETED")
                 .cache())
rdd_completed.count()

def rdd_job():
    return (rdd_completed
            .map(lambda t: (t["city"], (t["amount_mad"], 1)))
            .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
            .mapValues(lambda v: (v[0], v[1], v[0] / v[1]))
            .collect())

def df_job():
    return city_df_job().collect()

def sql_job():
    return spark.sql("""SELECT city, SUM(amount_mad), COUNT(*), AVG(amount_mad)
                        FROM tx_completed GROUP BY city""").collect()

for name, fn in [("RDD", rdd_job), ("DataFrame", df_job), ("SQL", sql_job)]:
    med, ts = bench(fn)
    print(f"{name:10s} médiane = {med:.3f} s   runs = {[round(t, 3) for t in ts]}")

RDD        médiane = 0.259 s   runs = [0.368, 0.312, 0.23, 0.259, 0.254]
DataFrame  médiane = 0.507 s   runs = [0.5, 0.529, 0.507, 0.353, 0.51]
SQL        médiane = 0.294 s   runs = [0.335, 0.301, 0.246, 0.253, 0.294]
